In [2]:
import ee

/Users/dan/tf_project_env/lib/python3.11/site-packages/google/api_core/_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.14) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/Users/dan/tf_project_env/lib/python3.11/site-packages/google/api_core/_python_package_support.py:206: FutureWarning: Package google.api_core depends on grpcio, currently installed at version 1.76.0. grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In October 2026, Google Cloud Python packages will raise their minimum requirements (including google-api-core, grpcio, and grpcio-status) to enforce grpcio >= 1.83.0. For more details on Google Cloud's post-quantum security migration, 

In [3]:
ee.Authenticate()


Successfully saved authorization token.


In [ ]:
import concurrent.futures
import os
import time
import ee
import numpy as np
import pandas as pd
from tqdm import tqdm

# -------------------------------------------------------------------------
# 1. Google Earth Engine Initialization
# -------------------------------------------------------------------------
# ee.Authenticate() # Run once if setting up a new environment
ee.Initialize()

# -------------------------------------------------------------------------
# 2. Load and Clean Raw Incident Data
# -------------------------------------------------------------------------
INPUT_FILE = "L005169-082426.xlsx"
CHECKPOINT_FILE = "wildfire_extracted_features_checkpoint.parquet"
FINAL_OUTPUT_FILE = "Texas_Wildfires_Aggregated_Full.csv"

print(f"Reading {INPUT_FILE}...")
df_raw = pd.read_excel(INPUT_FILE)

# Filter out bad locations, missing coordinates, or missing dates
valid_texas_mask = (
    (df_raw["BadLocation"] == 0)
    & (df_raw["Lat"].between(25.80, 36.60))
    & (df_raw["Long"].between(-106.70, -93.50))
    & (df_raw["StartDate"].notna())
)
df_clean = df_raw[valid_texas_mask].copy().reset_index(drop=True)
print(f"Loaded {len(df_raw)} records. Valid records to process: {len(df_clean)}")

# -------------------------------------------------------------------------
# 3. Define Earth Engine Static Image Assets
# -------------------------------------------------------------------------
# Topography: USGS NED 1-Arc-Second (~30m seamless continental US)
dem = ee.Image("USGS/NED")
slope = ee.Terrain.slope(dem)
aspect = ee.Terrain.aspect(dem)

terrain_stack = (
    dem.rename("elevation")
    .addBands(slope.rename("slope_deg"))
    .addBands(aspect.rename("aspect_deg"))
)

# Soil: ISRIC SoilGrids 250m v2.0 (0-5cm depth layer)
soil_clay = ee.Image("projects/soilgrids-isric/clay_mean").select(
    "clay_0-5cm_mean", "soil_clay_pct"
)
soil_sand = ee.Image("projects/soilgrids-isric/sand_mean").select(
    "sand_0-5cm_mean", "soil_sand_pct"
)
soil_bd = ee.Image("projects/soilgrids-isric/bdod_mean").select(
    "bdod_0-5cm_mean", "soil_bulk_density"
)
soil_stack = soil_clay.addBands(soil_sand).addBands(soil_bd)

static_landscape = terrain_stack.addBands(soil_stack)

# -------------------------------------------------------------------------
# 4. Single-Incident Point Extraction Worker
# -------------------------------------------------------------------------
def extract_environmental_record(row_dict):
    """Samples static terrain/soil and dynamic gridMET conditions at the ignition point."""
    inc_id = row_dict["ID_Incident"]
    lat = float(row_dict["Lat"])
    lon = float(row_dict["Long"])
    start_dt = pd.to_datetime(row_dict["StartDate"])

    geom = ee.Geometry.Point([lon, lat])

    # Date temporal window: 3-day antecedent drying + ignition date
    date_start = (start_dt - pd.Timedelta(days=3)).strftime("%Y-%m-%d")
    date_end = (start_dt + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    res = {"ID_Incident": inc_id}

    try:
        # Sample static landscape rasters (scale=100m)
        static_sample = (
            static_landscape.sample(region=geom, scale=100, geometries=False)
            .first()
            .toDictionary()
            .getInfo()
        )
        if static_sample:
            res.update(static_sample)

        # Dynamic Meteorology (gridMET 4km daily)
        gridmet_col = (
            ee.ImageCollection("IDAHO_EPSCOR/GRIDMET")
            .filterBounds(geom)
            .filterDate(date_start, date_end)
            .select(["tmmn", "tmmx", "rmin", "rmax", "vs", "bi", "erc", "pr"])
        )

        gridmet_mean = gridmet_col.mean()
        weather_sample = (
            gridmet_mean.sample(region=geom, scale=4000, geometries=False)
            .first()
            .toDictionary()
            .getInfo()
        )

        if weather_sample:
            # Convert Kelvin to Celsius
            if "tmmx" in weather_sample:
                weather_sample["temp_max_c"] = weather_sample["tmmx"] - 273.15
            if "tmmn" in weather_sample:
                weather_sample["temp_min_c"] = weather_sample["tmmn"] - 273.15
            res.update(weather_sample)

        return res

    except Exception as e:
        return {"ID_Incident": inc_id, "error": str(e)}


# -------------------------------------------------------------------------
# 5. Parallel Batch Execution with Checkpointing
# -------------------------------------------------------------------------
def run_aggregation_pipeline(
    df_to_process, batch_size=500, max_workers=10
):
    # Resume from checkpoint if it exists
    processed_ids = set()
    if os.path.exists(CHECKPOINT_FILE):
        checkpoint_df = pd.read_parquet(CHECKPOINT_FILE)
        processed_ids = set(checkpoint_df["ID_Incident"].unique())
        print(
            f"Found existing checkpoint. Resuming from {len(processed_ids)} processed records."
        )
    else:
        checkpoint_df = pd.DataFrame()

    unprocessed = df_to_process[
        ~df_to_process["ID_Incident"].isin(processed_ids)
    ].copy()
    total = len(unprocessed)
    print(f"Remaining records to extract: {total}")

    batch_buffer = []

    for start_idx in range(0, total, batch_size):
        chunk = unprocessed.iloc[start_idx : start_idx + batch_size]
        chunk_dicts = chunk.to_dict(orient="records")

        print(
            f"\nProcessing slice {start_idx} to {min(start_idx + batch_size, total)} of {total}..."
        )

        with concurrent.futures.ThreadPoolExecutor(
            max_workers=max_workers
        ) as executor:
            future_to_fire = {
                executor.submit(extract_environmental_record, row): row[
                    "ID_Incident"
                ]
                for row in chunk_dicts
            }
            for future in tqdm(
                concurrent.futures.as_completed(future_to_fire),
                total=len(chunk_dicts),
            ):
                batch_buffer.append(future.result())

        # Write to parquet checkpoint after each batch
        chunk_res_df = pd.DataFrame(batch_buffer)
        if not checkpoint_df.empty:
            checkpoint_df = pd.concat(
                [checkpoint_df, chunk_res_df], ignore_index=True
            ).drop_duplicates(subset=["ID_Incident"], keep="last")
        else:
            checkpoint_df = chunk_res_df

        checkpoint_df.to_parquet(CHECKPOINT_FILE, index=False)
        batch_buffer = []
        time.sleep(1)  # Brief buffer to manage GEE connection pool

    return checkpoint_df


# -------------------------------------------------------------------------
# 6. Execute and Join to Original Dataset
# -------------------------------------------------------------------------
if __name__ == "__main__":
    RUN_SUBSET = False  # Set to True for testing

    target_df = df_clean.head(100) if RUN_SUBSET else df_clean
    features_df = run_aggregation_pipeline(
        target_df, batch_size=500, max_workers=10
    )

    print("\nMerging extracted features with original incident records...")
    final_merged_df = df_raw.merge(features_df, on="ID_Incident", how="left")
    print(f"Saving merged output to {FINAL_OUTPUT_FILE}...")
    final_merged_df.to_csv(FINAL_OUTPUT_FILE, index=False)
    print("Aggregation complete!")

Reading L005169-082426.xlsx...
Loaded 285858 records. Valid records to process: 282687
Remaining records to extract: 282687

Processing slice 0 to 500 of 282687...


100%|██████████| 500/500 [00:05<00:00, 96.57it/s] 



Processing slice 500 to 1000 of 282687...


100%|██████████| 500/500 [00:03<00:00, 135.76it/s]



Processing slice 1000 to 1500 of 282687...


...


Processing slice 16000 to 16500 of 282687...


100%|██████████| 500/500 [00:03<00:00, 153.28it/s]



Processing slice 16500 to 17000 of 282687...


 20%|█▉        | 99/500 [00:21<01:25,  4.68it/s] 


KeyboardInterrupt: 

In [9]:
import pandas as pd
import ee

# 1. Ensure df_clean has the required geographic coordinate columns
if "df_clean_prepared.parquet" in os.listdir():
    df_clean = pd.read_parquet("df_clean_prepared.parquet")
else:
    df_raw = pd.read_excel("L005169-082426.xlsx")
    valid_texas_mask = (
        (df_raw["BadLocation"] == 0)
        & (df_raw["Lat"].between(25.80, 36.60))
        & (df_raw["Long"].between(-106.70, -93.50))
        & (df_raw["StartDate"].notna())
    )
    df_clean = df_raw[valid_texas_mask].copy().reset_index(drop=True)
    df_clean.to_parquet("df_clean_prepared.parquet", index=False)

# 2. Modern 10m DEM Terrain Stack (no deprecation warnings)
dem = ee.ImageCollection("USGS/3DEP/10m_collection").mosaic()
slope = ee.Terrain.slope(dem)
aspect = ee.Terrain.aspect(dem)

terrain_stack = (
    dem.rename("elevation")
    .addBands(slope.rename("slope_deg"))
    .addBands(aspect.rename("aspect_deg"))
)

# Soil Stack
soil_clay = ee.Image("projects/soilgrids-isric/clay_mean").select("clay_0-5cm_mean", "soil_clay_pct")
soil_sand = ee.Image("projects/soilgrids-isric/sand_mean").select("sand_0-5cm_mean", "soil_sand_pct")
soil_bd = ee.Image("projects/soilgrids-isric/bdod_mean").select("bdod_0-5cm_mean", "soil_bulk_density")
soil_stack = soil_clay.addBands(soil_sand).addBands(soil_bd)

static_landscape = terrain_stack.addBands(soil_stack)

# 3. Test extraction on the first incident
test_row = df_clean.iloc[0].to_dict()
test_output = extract_environmental_record(test_row)
print(test_output)

ArrowTypeError: ("Expected bytes, got a 'int' object", 'Conversion failed for column FireName with type object')

In [18]:
# Check why that specific row failed
failed_fire = df_00[df_00["ID_Incident"] == "4DC07FB0-B463-4D6E-A653-002D19E60DF"]
print(failed_fire[["ID_Incident", "error"]].to_dict(orient="records"))

[]


In [16]:
df_00 = pd.read_parquet("wildfire_extracted_features_checkpoint.parquet")

# Test a single row
test_row = df_00.head()
test_row


,ID_Incident,aspect_deg,elevation,slope_deg,soil_bulk_density,soil_clay_pct,soil_sand_pct,bi,erc,pr,rmax,rmin,tmmn,tmmx,vs,temp_max_c,temp_min_c,error
0,AEA0A967-E4C9-45C3-BD5F-0000D3C0A9DD,130.942886,344.277130,0.047491,1.56,22.2,26.1,64.458786,82.995712,0.000000,59.942520,15.898148,298.923096,314.560791,3.796030,41.410791,25.773096,None
1,4DC07FB0-B463-4D6E-A653-0002D19E60DF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Element.toDictionary: Parameter 'element' is r...
2,2727E1EE-E447-42E2-9B77-00037B1278F1,135.736145,104.224258,0.042446,1.42,16.2,33.7,26.852280,35.116325,0.186692,88.170471,35.545086,280.313660,295.816528,4.593959,22.666528,7.163660,None
3,1D46D20F-1A5D-4A1F-864C-0003ADA6AD66,96.883423,265.084381,0.035238,1.55,27.8,36.8,27.962414,36.351845,0.189663,93.305214,24.454102,262.392365,277.968933,4.585820,4.818933,-10.757635,None
4,E47D6494-9964-4F0A-B4F8-0002615830C7,116.449394,226.535248,0.086467,1.58,40.7,18.5,41.746780,39.132381,0.000000,91.064240,28.440571,276.570099,294.755432,5.019740,21.605432,3.420099,None


In [19]:
import pandas as pd

CHECKPOINT_FILE = "wildfire_extracted_features_checkpoint.parquet"

# Load current progress
df_cp = pd.read_parquet(CHECKPOINT_FILE)

# Keep only successfully extracted rows (drop the ones with errors)
if "error" in df_cp.columns:
    df_clean_cp = df_cp[df_cp["error"].isna()].copy()
    print(f"Purged {len(df_cp) - len(df_clean_cp)} error rows. Keeping {len(df_clean_cp)} valid rows.")
    df_clean_cp.to_parquet(CHECKPOINT_FILE, index=False)

Purged 364 error rows. Keeping 1636 valid rows.


In [12]:
import os
import pandas as pd
import ee

PARQUET_CACHE = "df_clean_prepared.parquet"
INPUT_FILE = "L005169-082426.xlsx"

# 1. Load or prepare df_clean with strict string casting
if os.path.exists(PARQUET_CACHE):
    print(f"Loading cached clean data from {PARQUET_CACHE}...")
    df_clean = pd.read_parquet(PARQUET_CACHE)
else:
    print(f"Reading {INPUT_FILE}...")
    df_raw = pd.read_excel(INPUT_FILE)
    
    valid_texas_mask = (
        (df_raw["BadLocation"] == 0)
        & (df_raw["Lat"].between(25.80, 36.60))
        & (df_raw["Long"].between(-106.70, -93.50))
        & (df_raw["StartDate"].notna())
    )
    df_clean = df_raw[valid_texas_mask].copy().reset_index(drop=True)

    # Cast mixed object columns (such as FireName) to string to satisfy PyArrow
    obj_cols = df_clean.select_dtypes(include=["object"]).columns
    df_clean[obj_cols] = df_clean[obj_cols].astype(str)

    print(f"Caching {len(df_clean)} records to {PARQUET_CACHE}...")
    df_clean.to_parquet(PARQUET_CACHE, index=False)

print(f"Dataset ready: {len(df_clean)} records.")

# 2. Modern 10m DEM Terrain Stack & Soil Stack
dem = ee.ImageCollection("USGS/3DEP/10m_collection").mosaic()
slope = ee.Terrain.slope(dem)
aspect = ee.Terrain.aspect(dem)

terrain_stack = (
    dem.rename("elevation")
    .addBands(slope.rename("slope_deg"))
    .addBands(aspect.rename("aspect_deg"))
)

# Soil: ISRIC SoilGrids 250m v2.0 (0-5cm depth layer)
soil_clay = ee.Image("projects/soilgrids-isric/clay_mean").select(
    ["clay_0-5cm_mean"], ["soil_clay_pct"]
)
soil_sand = ee.Image("projects/soilgrids-isric/sand_mean").select(
    ["sand_0-5cm_mean"], ["soil_sand_pct"]
)
soil_bd = ee.Image("projects/soilgrids-isric/bdod_mean").select(
    ["bdod_0-5cm_mean"], ["soil_bulk_density"]
)
soil_stack = soil_clay.addBands(soil_sand).addBands(soil_bd)

static_landscape = terrain_stack.addBands(soil_stack)

# 3. Test extraction on the first incident
test_row = df_clean.iloc[0].to_dict()
test_output = extract_environmental_record(test_row)
print("\nExtraction Test Result:")
print(test_output)

Loading cached clean data from df_clean_prepared.parquet...
Dataset ready: 282687 records.

Extraction Test Result:
{'ID_Incident': 'AEA0A967-E4C9-45C3-BD5F-0000D3C0A9DD', 'aspect_deg': 130.94288635253906, 'elevation': 344.2771301269531, 'slope_deg': 0.047490961849689484, 'soil_bulk_density': 156, 'soil_clay_pct': 222, 'soil_sand_pct': 261, 'bi': 64.45878601074219, 'erc': 82.99571228027344, 'pr': 0, 'rmax': 59.94252014160156, 'rmin': 15.898147583007812, 'tmmn': 298.923095703125, 'tmmx': 314.560791015625, 'vs': 3.796029567718506, 'temp_max_c': 41.41079101562502, 'temp_min_c': 25.773095703125023}


In [13]:
# 1. Modern 10m DEM Terrain Stack
dem = ee.ImageCollection("USGS/3DEP/10m_collection").mosaic()
slope = ee.Terrain.slope(dem)
aspect = ee.Terrain.aspect(dem)

terrain_stack = (
    dem.rename("elevation")
    .addBands(slope.rename("slope_deg"))
    .addBands(aspect.rename("aspect_deg"))
)

# 2. Corrected Soil Band Selection & Renaming
soil_clay = ee.Image("projects/soilgrids-isric/clay_mean").select(
    ["clay_0-5cm_mean"], ["soil_clay_pct"]
)
soil_sand = ee.Image("projects/soilgrids-isric/sand_mean").select(
    ["sand_0-5cm_mean"], ["soil_sand_pct"]
)
soil_bd = ee.Image("projects/soilgrids-isric/bdod_mean").select(
    ["bdod_0-5cm_mean"], ["soil_bulk_density"]
)
soil_stack = soil_clay.addBands(soil_sand).addBands(soil_bd)

static_landscape = terrain_stack.addBands(soil_stack)

# 3. Test extraction on the first incident
test_row = df_clean.iloc[0].to_dict()
test_output = extract_environmental_record(test_row)
print("\nExtraction Test Result:")
print(test_output)


Extraction Test Result:
{'ID_Incident': 'AEA0A967-E4C9-45C3-BD5F-0000D3C0A9DD', 'aspect_deg': 130.94288635253906, 'elevation': 344.2771301269531, 'slope_deg': 0.047490961849689484, 'soil_bulk_density': 156, 'soil_clay_pct': 222, 'soil_sand_pct': 261, 'bi': 64.45878601074219, 'erc': 82.99571228027344, 'pr': 0, 'rmax': 59.94252014160156, 'rmin': 15.898147583007812, 'tmmn': 298.923095703125, 'tmmx': 314.560791015625, 'vs': 3.796029567718506, 'temp_max_c': 41.41079101562502, 'temp_min_c': 25.773095703125023}


In [14]:
import concurrent.futures
import os
import time
import ee
import numpy as np
import pandas as pd
from tqdm import tqdm

# -------------------------------------------------------------------------
# 1. Google Earth Engine Initialization
# -------------------------------------------------------------------------
# ee.Authenticate()  # Run once if setting up in a new environment
ee.Initialize()

# -------------------------------------------------------------------------
# 2. File Paths & Data Preparation
# -------------------------------------------------------------------------
INPUT_FILE = "L005169-082426.xlsx"
PARQUET_PREPARED = "df_clean_prepared.parquet"
CHECKPOINT_FILE = "wildfire_extracted_features_checkpoint.parquet"
FINAL_OUTPUT_FILE = "Texas_Wildfires_Aggregated_Full.csv"

# Load or generate clean cached dataset
if os.path.exists(PARQUET_PREPARED):
    print(f"Loading cached clean incidents from {PARQUET_PREPARED}...")
    df_clean = pd.read_parquet(PARQUET_PREPARED)
    df_raw = pd.read_excel(INPUT_FILE)
else:
    print(f"Reading raw workbook: {INPUT_FILE}...")
    df_raw = pd.read_excel(INPUT_FILE)

    # Filter bad coordinates, missing dates, or bounding coordinates outside Texas
    valid_texas_mask = (
        (df_raw["BadLocation"] == 0)
        & (df_raw["Lat"].between(25.80, 36.60))
        & (df_raw["Long"].between(-106.70, -93.50))
        & (df_raw["StartDate"].notna())
    )
    df_clean = df_raw[valid_texas_mask].copy().reset_index(drop=True)

    # Cast object columns to string to prevent ArrowTypeError with mixed types
    obj_cols = df_clean.select_dtypes(include=["object"]).columns
    df_clean[obj_cols] = df_clean[obj_cols].astype(str)

    print(f"Caching {len(df_clean)} verified records to {PARQUET_PREPARED}...")
    df_clean.to_parquet(PARQUET_PREPARED, index=False)

print(f"Total raw incidents: {len(df_raw)} | Clean incidents to extract: {len(df_clean)}")

# -------------------------------------------------------------------------
# 3. Define Earth Engine Static Image Assets
# -------------------------------------------------------------------------
# Topography: USGS 3DEP 10m Collection (mosaicked for seamless coverage)
dem = ee.ImageCollection("USGS/3DEP/10m_collection").mosaic()
slope = ee.Terrain.slope(dem)
aspect = ee.Terrain.aspect(dem)

terrain_stack = (
    dem.rename("elevation")
    .addBands(slope.rename("slope_deg"))
    .addBands(aspect.rename("aspect_deg"))
)

# Soil: ISRIC SoilGrids 250m v2.0 (0-5cm depth layer)
soil_clay = ee.Image("projects/soilgrids-isric/clay_mean").select(
    ["clay_0-5cm_mean"], ["soil_clay_pct"]
)
soil_sand = ee.Image("projects/soilgrids-isric/sand_mean").select(
    ["sand_0-5cm_mean"], ["soil_sand_pct"]
)
soil_bd = ee.Image("projects/soilgrids-isric/bdod_mean").select(
    ["bdod_0-5cm_mean"], ["soil_bulk_density"]
)
soil_stack = soil_clay.addBands(soil_sand).addBands(soil_bd)

static_landscape = terrain_stack.addBands(soil_stack)

# -------------------------------------------------------------------------
# 4. Single-Incident Point Extraction Worker
# -------------------------------------------------------------------------
def extract_environmental_record(row_dict):
    """Samples static terrain/soil and dynamic gridMET conditions at the ignition point."""
    inc_id = row_dict["ID_Incident"]
    lat = float(row_dict["Lat"])
    lon = float(row_dict["Long"])
    start_dt = pd.to_datetime(row_dict["StartDate"])

    geom = ee.Geometry.Point([lon, lat])

    # Date temporal window: 3-day antecedent drying + ignition date
    date_start = (start_dt - pd.Timedelta(days=3)).strftime("%Y-%m-%d")
    date_end = (start_dt + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    res = {"ID_Incident": inc_id}

    try:
        # Sample static landscape rasters (scale=100m)
        static_sample = (
            static_landscape.sample(region=geom, scale=100, geometries=False)
            .first()
            .toDictionary()
            .getInfo()
        )
        if static_sample:
            # Apply SoilGrids integer-scaling factors to standard physical units
            if "soil_clay_pct" in static_sample:
                static_sample["soil_clay_pct"] = static_sample["soil_clay_pct"] / 10.0
            if "soil_sand_pct" in static_sample:
                static_sample["soil_sand_pct"] = static_sample["soil_sand_pct"] / 10.0
            if "soil_bulk_density" in static_sample:
                static_sample["soil_bulk_density"] = (
                    static_sample["soil_bulk_density"] / 100.0
                )
            res.update(static_sample)

        # Dynamic Meteorology: gridMET 4km daily
        gridmet_col = (
            ee.ImageCollection("IDAHO_EPSCOR/GRIDMET")
            .filterBounds(geom)
            .filterDate(date_start, date_end)
            .select(["tmmn", "tmmx", "rmin", "rmax", "vs", "bi", "erc", "pr"])
        )

        gridmet_mean = gridmet_col.mean()
        weather_sample = (
            gridmet_mean.sample(region=geom, scale=4000, geometries=False)
            .first()
            .toDictionary()
            .getInfo()
        )

        if weather_sample:
            # Convert Kelvin to Celsius
            if "tmmx" in weather_sample:
                weather_sample["temp_max_c"] = weather_sample["tmmx"] - 273.15
            if "tmmn" in weather_sample:
                weather_sample["temp_min_c"] = weather_sample["tmmn"] - 273.15
            res.update(weather_sample)

        return res

    except Exception as e:
        return {"ID_Incident": inc_id, "error": str(e)}

# -------------------------------------------------------------------------
# 5. Parallel Batch Execution with Checkpointing
# -------------------------------------------------------------------------
def run_aggregation_pipeline(df_to_process, batch_size=500, max_workers=10):
    processed_ids = set()
    if os.path.exists(CHECKPOINT_FILE):
        checkpoint_df = pd.read_parquet(CHECKPOINT_FILE)
        # Drop previous errors from checkpoint so failed records get retried
        if "error" in checkpoint_df.columns:
            checkpoint_df = checkpoint_df[checkpoint_df["error"].isna()].copy()
        processed_ids = set(checkpoint_df["ID_Incident"].unique())
        print(
            f"Found existing valid checkpoint. Resuming from {len(processed_ids)} completed records."
        )
    else:
        checkpoint_df = pd.DataFrame()

    unprocessed = df_to_process[
        ~df_to_process["ID_Incident"].isin(processed_ids)
    ].copy()
    total = len(unprocessed)
    print(f"Remaining records to extract: {total}")

    if total == 0:
        return checkpoint_df

    batch_buffer = []
    current_offset = len(processed_ids)

    for start_idx in range(0, total, batch_size):
        chunk = unprocessed.iloc[start_idx : start_idx + batch_size]
        chunk_dicts = chunk.to_dict(orient="records")

        abs_start = current_offset + start_idx
        abs_end = min(abs_start + len(chunk), len(df_to_process))
        print(f"\nProcessing slice {abs_start} to {abs_end} of {len(df_to_process)}...")

        with concurrent.futures.ThreadPoolExecutor(
            max_workers=max_workers
        ) as executor:
            future_to_fire = {
                executor.submit(extract_environmental_record, row): row[
                    "ID_Incident"
                ]
                for row in chunk_dicts
            }
            for future in tqdm(
                concurrent.futures.as_completed(future_to_fire),
                total=len(chunk_dicts),
            ):
                batch_buffer.append(future.result())

        # Commit to parquet checkpoint after each slice
        chunk_res_df = pd.DataFrame(batch_buffer)
        if not checkpoint_df.empty:
            checkpoint_df = pd.concat(
                [checkpoint_df, chunk_res_df], ignore_index=True
            ).drop_duplicates(subset=["ID_Incident"], keep="last")
        else:
            checkpoint_df = chunk_res_df

        checkpoint_df.to_parquet(CHECKPOINT_FILE, index=False)
        batch_buffer = []
        time.sleep(1)  # Brief connection pool cooldown

    return checkpoint_df

# -------------------------------------------------------------------------
# 6. Pipeline Execution and Final Left Merge
# -------------------------------------------------------------------------
if __name__ == "__main__":
    # Remove old corrupt checkpoint if you haven't already:
    # if os.path.exists(CHECKPOINT_FILE):
    #     os.remove(CHECKPOINT_FILE)

    RUN_SUBSET = False  # Set to True and inspect df_clean.head(50) if testing small slices
    target_df = df_clean.head(100) if RUN_SUBSET else df_clean

    features_df = run_aggregation_pipeline(
        target_df, batch_size=500, max_workers=10
    )

    print("\nMerging extracted features with original incident records...")
    final_merged_df = df_raw.merge(features_df, on="ID_Incident", how="left")

    print(f"Saving finalized aggregated dataset to {FINAL_OUTPUT_FILE}...")
    final_merged_df.to_csv(FINAL_OUTPUT_FILE, index=False)
    print("Aggregation complete!")

Reading raw workbook: L005169-082426.xlsx...
Caching 282687 verified records to df_clean_prepared.parquet...
Total raw incidents: 285858 | Clean incidents to extract: 282687
Remaining records to extract: 282687

Processing slice 0 to 500 of 282687...


100%|██████████| 500/500 [00:19<00:00, 26.08it/s]



Processing slice 500 to 1000 of 282687...


100%|██████████| 500/500 [00:17<00:00, 29.17it/s]



Processing slice 1000 to 1500 of 282687...


...


Processing slice 1500 to 2000 of 282687...


100%|██████████| 500/500 [00:15<00:00, 31.52it/s]



Processing slice 2000 to 2500 of 282687...


 83%|████████▎ | 413/500 [00:13<00:02, 30.77it/s]


KeyboardInterrupt: 

In [20]:
import concurrent.futures
import os
import time
import ee
import numpy as np
import pandas as pd
from tqdm import tqdm

# -------------------------------------------------------------------------
# 1. Google Earth Engine Initialization
# -------------------------------------------------------------------------
# ee.Authenticate()  # Run once if needed
ee.Initialize()

# -------------------------------------------------------------------------
# 2. File Paths & Data Loading
# -------------------------------------------------------------------------
INPUT_FILE = "L005169-082426.xlsx"
PARQUET_PREPARED = "df_clean_prepared.parquet"
CHECKPOINT_FILE = "wildfire_extracted_features_checkpoint.parquet"
FINAL_OUTPUT_FILE = "Texas_Wildfires_Aggregated_Full.csv"

# Fast-load preprocessed parquet if available, otherwise process raw Excel
if os.path.exists(PARQUET_PREPARED):
    print(f"Loading cached incidents from {PARQUET_PREPARED}...")
    df_clean = pd.read_parquet(PARQUET_PREPARED)
    df_raw = pd.read_excel(INPUT_FILE)
else:
    print(f"Reading raw workbook: {INPUT_FILE}...")
    df_raw = pd.read_excel(INPUT_FILE)
    valid_texas_mask = (
        (df_raw["BadLocation"] == 0)
        & (df_raw["Lat"].between(25.80, 36.60))
        & (df_raw["Long"].between(-106.70, -93.50))
        & (df_raw["StartDate"].notna())
    )
    df_clean = df_raw[valid_texas_mask].copy().reset_index(drop=True)
    obj_cols = df_clean.select_dtypes(include=["object"]).columns
    df_clean[obj_cols] = df_clean[obj_cols].astype(str)
    df_clean.to_parquet(PARQUET_PREPARED, index=False)

print(f"Total raw incidents: {len(df_raw)} | Valid incidents to process: {len(df_clean)}")

# -------------------------------------------------------------------------
# 3. Define Earth Engine Static Image Assets
# -------------------------------------------------------------------------
# Topography: USGS 3DEP 10m Collection
dem = ee.ImageCollection("USGS/3DEP/10m_collection").mosaic()
slope = ee.Terrain.slope(dem)
aspect = ee.Terrain.aspect(dem)

terrain_stack = (
    dem.rename("elevation")
    .addBands(slope.rename("slope_deg"))
    .addBands(aspect.rename("aspect_deg"))
)

# Soil: ISRIC SoilGrids 250m v2.0 (0-5cm depth layer)
soil_clay = ee.Image("projects/soilgrids-isric/clay_mean").select(
    ["clay_0-5cm_mean"], ["soil_clay_pct"]
)
soil_sand = ee.Image("projects/soilgrids-isric/sand_mean").select(
    ["sand_0-5cm_mean"], ["soil_sand_pct"]
)
soil_bd = ee.Image("projects/soilgrids-isric/bdod_mean").select(
    ["bdod_0-5cm_mean"], ["soil_bulk_density"]
)
soil_stack = soil_clay.addBands(soil_sand).addBands(soil_bd)

static_landscape = terrain_stack.addBands(soil_stack)

# -------------------------------------------------------------------------
# 4. Single-Incident Point Extraction Worker (SAFE NULL GUARDS)
# -------------------------------------------------------------------------
def extract_environmental_record(row_dict):
    """Samples static terrain/soil and dynamic gridMET conditions with safe null guards."""
    inc_id = row_dict["ID_Incident"]
    lat = float(row_dict["Lat"])
    lon = float(row_dict["Long"])
    start_dt = pd.to_datetime(row_dict["StartDate"])

    geom = ee.Geometry.Point([lon, lat])

    # Date temporal window: 3-day antecedent drying + ignition date
    date_start = (start_dt - pd.Timedelta(days=3)).strftime("%Y-%m-%d")
    date_end = (start_dt + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    res = {"ID_Incident": inc_id}

    try:
        # 1. Sample Static Rasters safely
        static_fc = static_landscape.sample(region=geom, scale=100, geometries=False).getInfo()
        if static_fc and static_fc.get("features"):
            static_props = static_fc["features"][0].get("properties", {})
            if "soil_clay_pct" in static_props:
                static_props["soil_clay_pct"] = static_props["soil_clay_pct"] / 10.0
            if "soil_sand_pct" in static_props:
                static_props["soil_sand_pct"] = static_props["soil_sand_pct"] / 10.0
            if "soil_bulk_density" in static_props:
                static_props["soil_bulk_density"] = static_props["soil_bulk_density"] / 100.0
            res.update(static_props)

        # 2. Dynamic Meteorology (gridMET 1979 - Present)
        if start_dt.year >= 1979:
            gridmet_col = (
                ee.ImageCollection("IDAHO_EPSCOR/GRIDMET")
                .filterBounds(geom)
                .filterDate(date_start, date_end)
                .select(["tmmn", "tmmx", "rmin", "rmax", "vs", "bi", "erc", "pr"])
            )

            gridmet_mean = gridmet_col.mean()
            weather_fc = gridmet_mean.sample(region=geom, scale=4000, geometries=False).getInfo()
            if weather_fc and weather_fc.get("features"):
                weather_props = weather_fc["features"][0].get("properties", {})
                if "tmmx" in weather_props:
                    weather_props["temp_max_c"] = weather_props["tmmx"] - 273.15
                if "tmmn" in weather_props:
                    weather_props["temp_min_c"] = weather_props["tmmn"] - 273.15
                res.update(weather_props)

        return res

    except Exception as e:
        return {"ID_Incident": inc_id, "error": str(e)}

# -------------------------------------------------------------------------
# 5. Parallel Batch Execution with Checkpointing
# -------------------------------------------------------------------------
def run_aggregation_pipeline(df_to_process, batch_size=500, max_workers=10):
    processed_ids = set()
    if os.path.exists(CHECKPOINT_FILE):
        checkpoint_df = pd.read_parquet(CHECKPOINT_FILE)
        if "error" in checkpoint_df.columns:
            checkpoint_df = checkpoint_df[checkpoint_df["error"].isna()].copy()
        processed_ids = set(checkpoint_df["ID_Incident"].unique())
        print(f"Resuming from {len(processed_ids)} previously validated records.")
    else:
        checkpoint_df = pd.DataFrame()

    unprocessed = df_to_process[~df_to_process["ID_Incident"].isin(processed_ids)].copy()
    total = len(unprocessed)
    print(f"Remaining records to extract: {total}")

    if total == 0:
        return checkpoint_df

    batch_buffer = []
    current_offset = len(processed_ids)

    for start_idx in range(0, total, batch_size):
        chunk = unprocessed.iloc[start_idx : start_idx + batch_size]
        chunk_dicts = chunk.to_dict(orient="records")

        abs_start = current_offset + start_idx
        abs_end = min(abs_start + len(chunk), len(df_to_process))
        print(f"\nProcessing slice {abs_start} to {abs_end} of {len(df_to_process)}...")

        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_fire = {
                executor.submit(extract_environmental_record, row): row["ID_Incident"]
                for row in chunk_dicts
            }
            for future in tqdm(concurrent.futures.as_completed(future_to_fire), total=len(chunk_dicts)):
                batch_buffer.append(future.result())

        # Save checkpoint to parquet after every slice
        chunk_res_df = pd.DataFrame(batch_buffer)
        if not checkpoint_df.empty:
            checkpoint_df = pd.concat([checkpoint_df, chunk_res_df], ignore_index=True).drop_duplicates(
                subset=["ID_Incident"], keep="last"
            )
        else:
            checkpoint_df = chunk_res_df

        checkpoint_df.to_parquet(CHECKPOINT_FILE, index=False)
        batch_buffer = []
        time.sleep(1)

    return checkpoint_df

# -------------------------------------------------------------------------
# 6. Pipeline Execution and Final Join
# -------------------------------------------------------------------------
if __name__ == "__main__":
    features_df = run_aggregation_pipeline(df_clean, batch_size=500, max_workers=10)

    print("\nMerging extracted features with original incident records...")
    final_merged_df = df_raw.merge(features_df, on="ID_Incident", how="left")

    print(f"Saving finalized aggregated dataset to {FINAL_OUTPUT_FILE}...")
    final_merged_df.to_csv(FINAL_OUTPUT_FILE, index=False)
    print("Aggregation complete!")

Loading cached incidents from df_clean_prepared.parquet...


KeyboardInterrupt: 

In [24]:
# 1. Clean out the old error rows from the existing checkpoint
import pandas as pd

CHECKPOINT_FILE = "wildfire_extracted_features_checkpoint.parquet"
df_cp = pd.read_parquet(CHECKPOINT_FILE)

if "error" in df_cp.columns:
    df_clean_cp = df_cp[df_cp["error"].isna()].copy()
    print(f"Purged {len(df_cp) - len(df_clean_cp)} errored rows from checkpoint.")
    df_clean_cp.to_parquet(CHECKPOINT_FILE, index=False)

# 2. Test the single incident that failed previously (from df_clean)
# Find the exact row that failed
test_record = df_clean[df_clean["ID_Incident"].str.startswith("4DC07FB0")].iloc[0].to_dict()

# Run the updated Section 4 function on it
test_result = extract_environmental_record(test_record)
print("\nTest Result for previously failed fire:")
print(test_result)

Purged 0 errored rows from checkpoint.

Test Result for previously failed fire:
{'ID_Incident': '4DC07FB0-B463-4D6E-A653-0002D19E60DF', 'bi': 29.972858428955078, 'erc': 35.01403045654297, 'pr': 0.08352009952068329, 'rmax': 88.50794219970703, 'rmin': 26.14594268798828, 'tmmn': 278.07684326171875, 'tmmx': 293.1996154785156, 'vs': 2.7440547943115234, 'temp_max_c': 20.049615478515648, 'temp_min_c': 4.926843261718773}


In [23]:
def extract_environmental_record(row_dict):
    """Samples static terrain/soil and dynamic gridMET conditions with safe null guards."""
    inc_id = row_dict["ID_Incident"]
    lat = float(row_dict["Lat"])
    lon = float(row_dict["Long"])
    start_dt = pd.to_datetime(row_dict["StartDate"])

    geom = ee.Geometry.Point([lon, lat])

    # Date temporal window: 3-day antecedent drying + ignition date
    date_start = (start_dt - pd.Timedelta(days=3)).strftime("%Y-%m-%d")
    date_end = (start_dt + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    res = {"ID_Incident": inc_id}

    try:
        # 1. Sample Static Rasters safely (Notice: NO .toDictionary() call!)
        static_fc = static_landscape.sample(region=geom, scale=100, geometries=False).getInfo()
        if static_fc and static_fc.get("features"):
            static_props = static_fc["features"][0].get("properties", {})
            if "soil_clay_pct" in static_props:
                static_props["soil_clay_pct"] = static_props["soil_clay_pct"] / 10.0
            if "soil_sand_pct" in static_props:
                static_props["soil_sand_pct"] = static_props["soil_sand_pct"] / 10.0
            if "soil_bulk_density" in static_props:
                static_props["soil_bulk_density"] = static_props["soil_bulk_density"] / 100.0
            res.update(static_props)

        # 2. Dynamic Meteorology (gridMET 1979 - Present)
        if start_dt.year >= 1979:
            gridmet_col = (
                ee.ImageCollection("IDAHO_EPSCOR/GRIDMET")
                .filterBounds(geom)
                .filterDate(date_start, date_end)
                .select(["tmmn", "tmmx", "rmin", "rmax", "vs", "bi", "erc", "pr"])
            )

            gridmet_mean = gridmet_col.mean()
            weather_fc = gridmet_mean.sample(region=geom, scale=4000, geometries=False).getInfo()
            if weather_fc and weather_fc.get("features"):
                weather_props = weather_fc["features"][0].get("properties", {})
                if "tmmx" in weather_props:
                    weather_props["temp_max_c"] = weather_props["tmmx"] - 273.15
                if "tmmn" in weather_props:
                    weather_props["temp_min_c"] = weather_props["tmmn"] - 273.15
                res.update(weather_props)

        return res

    except Exception as e:
        return {"ID_Incident": inc_id, "error": str(e)}

In [25]:
import pandas as pd

CHECKPOINT_FILE = "wildfire_extracted_features_checkpoint.parquet"
df_cp = pd.read_parquet(CHECKPOINT_FILE)

# Valid extractions will always have weather data (e.g. 'bi' is not null)
initial_count = len(df_cp)
df_clean_cp = df_cp[df_cp["bi"].notna()].copy()

# Drop the 'error' column if it exists
if "error" in df_clean_cp.columns:
    df_clean_cp = df_clean_cp.drop(columns=["error"])

purged_count = initial_count - len(df_clean_cp)
print(f"Purged {purged_count} failed/incomplete rows. Retaining {len(df_clean_cp)} clean rows.")

# Overwrite checkpoint with only valid rows
df_clean_cp.to_parquet(CHECKPOINT_FILE, index=False)

Purged 0 failed/incomplete rows. Retaining 1636 clean rows.


In [26]:
import pandas as pd

CHECKPOINT_FILE = "wildfire_extracted_features_checkpoint.parquet"
df_cp = pd.read_parquet(CHECKPOINT_FILE)

# Retain only records that successfully captured features (e.g., 'bi' is populated)
initial_count = len(df_cp)
df_clean_cp = df_cp[df_cp["bi"].notna()].copy()

# Drop the temporary error column if present
if "error" in df_clean_cp.columns:
    df_clean_cp = df_clean_cp.drop(columns=["error"])

purged_count = initial_count - len(df_clean_cp)
print(f"Purged {purged_count} failed rows. Retained {len(df_clean_cp)} valid rows.")

# Overwrite checkpoint with only clean records
df_clean_cp.to_parquet(CHECKPOINT_FILE, index=False)

Purged 0 failed rows. Retained 1636 valid rows.


In [29]:
features_df = run_aggregation_pipeline(df_clean, batch_size=500, max_workers=10)

print("\nMerging extracted features with original incident records...")
final_merged_df = df_raw.merge(features_df, on="ID_Incident", how="left")

print(f"Saving finalized aggregated dataset to {FINAL_OUTPUT_FILE}...")
final_merged_df.to_csv(FINAL_OUTPUT_FILE, index=False)
print("Aggregation complete!")

Found existing valid checkpoint. Resuming from 276636 completed records.
Remaining records to extract: 6051

Processing slice 276636 to 277136 of 282687...


100%|██████████| 500/500 [00:08<00:00, 57.75it/s]



Processing slice 277136 to 277636 of 282687...


100%|██████████| 500/500 [00:14<00:00, 34.64it/s]



Processing slice 277636 to 278136 of 282687...


...


Processing slice 282136 to 282636 of 282687...


100%|██████████| 500/500 [00:13<00:00, 37.52it/s]



Processing slice 282636 to 282687 of 282687...


100%|██████████| 51/51 [00:01<00:00, 33.89it/s]



Merging extracted features with original incident records...
Saving finalized aggregated dataset to Texas_Wildfires_Aggregated_Full.csv...
Aggregation complete!
